In [ ]:
import requests

URL = "https://incidentdatabase.ai/api/graphql"

introspection_query = """
{
  __type(name: "Incident") {
    name
    fields {
      name
      type { name kind ofType { name kind } }
    }
  }
}
"""

# The AIID GraphQL endpoint rejects POST requests whose Origin/Referer header
# isn't https://incidentdatabase.ai or https://staging-aiid.netlify.app
# (see site/gatsby-site/netlify/functions/graphql.ts in responsible-ai-collaborative/aiid).
# That check only applies to POST — GET requests bypass it, so we send the
# query as a URL parameter instead.
r = requests.get(URL, params={"query": introspection_query})
print(r.json())

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import glob
import json
import os

# Folder in Google Drive containing the batch JSON files to combine.
# Adjust this path to match where your batch files actually live.
BATCH_DIR = "/content/drive/MyDrive/google"
BATCH_PATTERN = "batch*.json"
OUTPUT_JSON = os.path.join(BATCH_DIR, "combined.json")
OUTPUT_JSONL = os.path.join(BATCH_DIR, "combined.jsonl")

batch_paths = sorted(glob.glob(os.path.join(BATCH_DIR, BATCH_PATTERN)))
print(f"Found {len(batch_paths)} batch file(s)")

combined = []

for path in batch_paths:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Each batch file may be a JSON array of records, or a single object.
    if isinstance(data, list):
        combined.extend(data)
    else:
        combined.append(data)

    print(f"  {os.path.basename(path)}: {len(data) if isinstance(data, list) else 1} record(s)")

print(f"Total combined records: {len(combined)}")

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(combined, f, ensure_ascii=False, indent=2)

with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for record in combined:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Wrote {OUTPUT_JSON}")
print(f"Wrote {OUTPUT_JSONL}")